In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
from src.model import radius_model # load model functions

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit


In [15]:
df = pd.read_csv("../data/processed/bubble_curves.csv")
print(df)

        time_s  radius_um condition_id
0     0.000000  15.000000        f0_93
1    10.040053  11.977064        f0_93
2    14.098798   9.986239        f0_93
3    20.293725   7.931193        f0_93
4    27.556742   4.045872        f0_93
5     0.000000  15.000000           f1
6    10.360481  15.003440           f1
7    35.781041  14.802752           f1
8    55.754339  14.794725           f1
9    84.379172  14.802752           f1
10  106.381842  14.891055           f1
11  125.767690  15.099771           f1
12  145.847797  14.987385           f1
13    0.000000  15.000000        f0_84
14    5.340454  11.912844        f0_84
15   10.040053   6.967890        f0_84
16   12.016021   4.944954        f0_84
17   13.244326   2.440367        f0_84
18    0.000000  15.000000        f0_98
19   15.380507  13.951835        f0_98
20   62.002670  10.636468        f0_98
21   87.369826   8.725917        f0_98
22  114.018692   6.422018        f0_98
23  133.404539   5.017202        f0_98
24  148.090788   2.408257

In [22]:
def radius_model(t, k):
    return np.sqrt(np.maximum(R0**2 - 2*k*t, 0))

In [25]:
# Define the model function for curve fitting
# The initial radius R0 is known 
R0 = df['radius_um'][0]  
def radius_model(t, k):
    return np.sqrt(np.maximum(R0**2 - 2*k*t, 0))

# Fit the first condition, make it work, then automate all conditions.
condition_names = df['condition_id'].unique()

df_results = pd.DataFrame(columns=['condition_id', 'R0_um', 'k_fit', 'RMSE'])

for cond in df['condition_id'].unique():

    data = df[df['condition_id'] == cond]

    t = data['time_s'].values
    r = data['radius_um'].values

    k_guess = 0.01

    popt, _ = curve_fit(radius_model, t, r, p0=[k_guess])

    k_fit = popt[0]

    y_pred = radius_model(t, k_fit)

    rmse = np.sqrt(np.mean((r - y_pred)**2))

    df_results.loc[len(df_results)] = [cond, R0, k_fit, rmse]

print(df_results)


  condition_id  R0_um     k_fit      RMSE
0        f0_93   15.0  4.087216  1.825847
1           f1   15.0  0.008953  0.122684
2        f0_84   15.0  8.300539  0.330386
3        f0_98   15.0  0.772432  1.051478
